# Laboratório de Voz — Estúdio IA | Qwen3-TTS gratuito

**Objetivo:** produzir cinco amostras curtas de documentário (português, inglês, espanhol, francês e alemão) com o modelo aberto `Qwen3-TTS-12Hz-0.6B-CustomVoice`.

**Custo de API:** zero. **Execução:** Google Colab (GPU gratuita quando disponível, mas sem garantia de disponibilidade ou uso ilimitado). **Não conecta nem modifica o Estúdio do Railway.**

### Primeiro
1. Em **Ambiente de execução → Alterar o tipo de ambiente de execução**, selecione **T4 GPU** (ou outra GPU gratuita).
2. Selecione **Ambiente de execução → Executar tudo**. Confirme a execução das células quando o Colab solicitar.
3. No final, ouça os cinco arquivos gerados e baixe o ZIP.

**Limitação importante:** o modelo suporta os cinco idiomas, porém seus locutores predefinidos são nativos principalmente de inglês, chinês, coreano e japonês; avalie o sotaque de cada idioma. O modelo tem licença Apache 2.0. Não use clonagem de voz sem autorização.

Fonte: https://huggingface.co/Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice

In [ ]:
import torch
assert torch.cuda.is_available(), ("GPU não encontrada. No menu do Colab, escolha Ambiente de execução "
    "> Alterar o tipo de ambiente de execução > T4 GPU, depois execute novamente.")
gpu = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
# GPUs T4 não suportam bfloat16. FP32 é mais conservador para evitar erros numéricos.
precision = torch.bfloat16 if cap[0] >= 8 else torch.float32
print(f"GPU: {gpu} | capacidade={cap} | precisão={precision}")

In [ ]:
%pip install -q -U "qwen-tts" "soundfile" "numpy"

In [ ]:
from qwen_tts import Qwen3TTSModel
from IPython.display import display, Audio
import soundfile as sf
import numpy as np
import json, shutil, time
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice"
print("Carregando Qwen3-TTS 0.6B. Na primeira execução, ele baixa os pesos do modelo.")
model = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map="cuda:0",
    dtype=precision,
    attn_implementation="sdpa",
)
print("Modelo carregado. Locutores:", model.get_supported_speakers())
print("Idiomas:", model.get_supported_languages())

### Gerar as cinco vozes

A voz inicial é **Aiden** (masculina, originalmente inglesa). Podemos testar **Ryan** depois, alterando `SPEAKER` e executando somente a próxima célula novamente. Não confundiremos a capacidade de falar um idioma com sotaque nativo.

In [ ]:
SPEAKER = "Aiden"  # alternativas: "Ryan" (inglês) e "Uncle_Fu" (chinês)
AMOSTRAS = {
    "pt": ("Portuguese",
        "Existe uma força silenciosa em quem aprende a estabelecer limites. Nem toda decisão precisa ser explicada, e nem toda expectativa precisa ser atendida. Hoje, vamos explorar como a serenidade pode transformar escolhas difíceis em liberdade."),
    "en": ("English",
        "There is a quiet strength in learning to set boundaries. Not every decision needs an explanation, and not every expectation deserves an answer. Today, we explore how a calm mind can turn difficult choices into genuine freedom."),
    "es": ("Spanish",
        "Existe una fuerza silenciosa en quien aprende a poner límites. No todas las decisiones necesitan una explicación, ni todas las expectativas merecen una respuesta. Hoy descubriremos cómo la serenidad puede convertir una elección difícil en libertad."),
    "fr": ("French",
        "Il existe une force tranquille chez ceux qui apprennent à poser des limites. Chaque décision ne demande pas d'explication, et chaque attente ne mérite pas de réponse. Aujourd'hui, découvrons comment le calme peut transformer un choix difficile en véritable liberté."),
    "de": ("German",
        "Wer lernt, klare Grenzen zu setzen, entdeckt eine stille Stärke. Nicht jede Entscheidung braucht eine Erklärung, und nicht jede Erwartung verdient eine Antwort. Heute sehen wir, wie innere Ruhe schwierige Entscheidungen in echte Freiheit verwandeln kann."),
}
PASTA = Path("/content/amostras_estudio_ia_qwen3")
PASTA.mkdir(parents=True, exist_ok=True)
resultados = []
for sigla, (idioma, texto) in AMOSTRAS.items():
    print(f"\nGerando {idioma} ({sigla}) com {SPEAKER} ...")
    antes = time.monotonic()
    try:
        with torch.inference_mode():
            audios, sr = model.generate_custom_voice(
                text=texto,
                language=idioma,
                speaker=SPEAKER,
            )
        wav = np.asarray(audios[0], dtype=np.float32)
        if len(wav) < int(sr):
            raise RuntimeError("Áudio mais curto que um segundo")
        nome = f"{sigla}_{SPEAKER.lower()}.wav"
        sf.write(str(PASTA / nome), wav, sr)
        segundos = round(len(wav) / sr, 1)
        print(f"{nome}: {segundos} segundos (processamento: {time.monotonic()-antes:.1f}s)")
        display(Audio(wav, rate=sr))
        resultados.append({"idioma":sigla,"speaker":SPEAKER,"arquivo":nome,
                           "duracao_s":segundos,"status":"ok"})
    except Exception as e:
        print("Falha neste idioma:", type(e).__name__, str(e)[:160])
        resultados.append({"idioma":sigla,"speaker":SPEAKER,"status":"erro",
                           "detalhes":type(e).__name__})
(PASTA / "resultado.json").write_text(
    json.dumps({"modelo":MODEL_ID,"testes":resultados},ensure_ascii=False,indent=2),
    encoding="utf-8")
print("\nConcluído:",sum(x["status"]=="ok" for x in resultados),"/5 idiomas.")

### Baixar resultados

O ZIP contém um WAV por idioma gerado com sucesso e um resumo das durações. Escute **naturalidade, pronúncia, sotaque, pausas e articulação**. Se a primeira voz não agradar, altere `SPEAKER` e execute a célula anterior novamente.

In [ ]:
from google.colab import files
arquivo = shutil.make_archive("/content/teste_qwen3_estudio_ia", "zip", root_dir=str(PASTA))
print("Arquivo pronto:", arquivo)
files.download(arquivo)

### Depois do teste

Este notebook apenas compara vozes. **Não é um serviço de narração 24 horas por dia** e não usa Railway. Se aprovarmos a qualidade, o próximo passo será um motor separado (local ou GPU dedicada) para gerar narrações em lote, com fila, checkpoints e integração segura ao Estúdio. GPUs gratuitas do Colab podem ficar indisponíveis e os notebooks expiram; não são infraestrutura de produção contínua.

Código e modelo oficiais: https://github.com/QwenLM/Qwen3-TTS

Políticas do Colab gratuito: https://research.google.com/colaboratory/intl/pt-BR/faq.html